In [ ]:
# ================================================================
#             📌 BIODIVERSITY PROCESSING FULL PIPELINE
# ================================================================

# ---  Préambule : imports ---
%matplotlib inline

import os, sys, importlib
from pathlib import Path
import ipynbname
import geopandas as gpd
import pandas as pd

# ================================================================
# 1️⃣  DEFINE GLOBAL PARAMETERS (only here!)
# ================================================================
COUNTRIES = ["Cambodia","Laos","Vietnam","Thailand"]              # ⚠️ List of countries to process
SOURCE = "GBIF"                       # 'GBIF' or 'INPN'
NAME_ATTRIBUTE = "sovereignt"
CODE_ATTRIBUTE = "sov_a3"
CELL_SIZES_KM = [10]  # Grid sizes to generate
TOLERANCE_GEOM = 0                    # Simplification tolerance
CLE_ID = "speciesID"
ANNEE_MIN = 1950
CHUNKSIZE = 10_000_000
SEUIL_OBS = 500                   # Fusion threshold (sum obs)
SEUIL_SPECIES=50

# ================================================================
# 2️⃣ GLOBAL PATH CONFIGURATION
# ================================================================
base_path = Path(ipynbname.path()).parent.parent.parent
path_data = base_path / 'Data'
scripts_path = base_path / 'Code' / 'Formatage' / 'scripts'
sys.path.append(str(scripts_path.resolve()))

print("📌 **CONFIGURATION DES CHEMINS**")
print(f"📍 base_path    : {base_path}")
print(f"📂 path_data     : {path_data}")
print(f"📂 scripts_path  : {scripts_path}")

# ================================================================
# 3️⃣ IMPORT CUSTOM MODULES
# ================================================================
import formatage_geo, formatage_biodiv, formatage_ObsToGrid, formatage_fusion
importlib.reload(formatage_geo)
importlib.reload(formatage_biodiv)
importlib.reload(formatage_ObsToGrid)
importlib.reload(formatage_fusion)

from formatage_geo import generate_country_grid, plot_country_grid, simplify_world_geometry, save_country_grid
from formatage_biodiv import read_biodiv_chunks, save_clean_biodiv
from formatage_ObsToGrid import assign_grid_to_points, generate_taxo_dict, aggregate_by_grid_species, save_processed_biodiv
from formatage_fusion import fusion_cells_by_obs, apply_fusion_to_biodiv, plot_fusionned_grid, save_fusionned_biodiv,save_merged_grid,fusion_cells_by_obs_and_species

# ================================================================
# DEFINE REUSABLE FUNCTION
# ================================================================
def process_country(zone):
    print(f"\n\n🚀🚀🚀 === TRAITEMENT : {zone.upper()} === 🚀🚀🚀")

    # ---------- LOAD GLOBAL SHAPE ----------
    geo_file = path_data/'SIG_global'/'world_boundaries.json'
    geo_gdf = gpd.read_file(os.path.join(geo_file))
    geo_gdf = simplify_world_geometry(geo_gdf, tolerance=TOLERANCE_GEOM)

    # ---------- 1) GENERATE GRIDS ----------
    output_grid = path_data / "SIG_zone" / zone
    output_grid.mkdir(parents=True, exist_ok=True)

    for cell_size_km in CELL_SIZES_KM:
        cle_geo = f"cellID_{cell_size_km}km"
        grid = generate_country_grid(
            world_gdf=geo_gdf,
            country_name=zone,
            name_attribute=NAME_ATTRIBUTE,
            grid_size_km=cell_size_km,
            cle_geo=cle_geo,
            code_attribute=CODE_ATTRIBUTE
        )
        save_country_grid(grid, output_grid, zone, cle_geo)
        print(f"📌 Grille {cell_size_km} km OK")
    
    # Display only the smallest grid
    plot_country_grid(geo_gdf, grid, NAME_ATTRIBUTE, zone)

    # ---------- 2) CLEAN BIODIV DATA ----------
    path_fichier = path_data / SOURCE / "raw" / f"{SOURCE}_{zone}.csv"

    print(f"📥 Nettoyage biodiversité {SOURCE}...")
    df_biodiv_clean = read_biodiv_chunks(
        path_fichier=path_fichier,
        source=SOURCE,
        cle_ID=CLE_ID,
        annee_min=ANNEE_MIN,
        chunksize=CHUNKSIZE
    )
    save_clean_biodiv(df_biodiv_clean, SOURCE, zone, path_data)
    print("👍 Biodiv cleaned")

    # ---------- 3) ASSIGN OBS TO GRID ----------
    cell_size_km = min(CELL_SIZES_KM)  # choose smallest cell
    cle_geo = f"cellID_{cell_size_km}km"
    file_path_grid= output_grid / f"{zone}_{cle_geo}.geojson"
    grid = gpd.read_file(os.path.join(file_path_grid))

    df_biodiv_clean = pd.read_csv(path_data / SOURCE / "clean" / f"{SOURCE}_{zone}.csv", dtype={CLE_ID: "string"})
    df_with_grid = assign_grid_to_points(df_biodiv_clean, grid, cle_geo)

    # add vernacular names
    dico_noms_vernaculaires = pd.read_csv(path_data/"Taxonomie"/ "dico_noms_vernaculaires_merged.csv")
    df_with_grid = pd.merge(df_with_grid, dico_noms_vernaculaires, on=CLE_ID, how="left")

    # aggregate
    dico_taxo = generate_taxo_dict(df_with_grid, CLE_ID)
    df_final = aggregate_by_grid_species(df_with_grid, cle_geo, dico_taxo, cle_ID=CLE_ID)
    save_processed_biodiv(df_final, SOURCE, zone, cle_geo, path_data)

    print("📌 Obs → Grid OK")

    # ---------- 4) FUSION CELLS ----------
    merged_fusion = fusion_cells_by_obs_and_species(df_final, grid, cle_geo, var_obs='nombreObs', seuil=SEUIL_OBS,cle_ID=CLE_ID,min_species=SEUIL_SPECIES,methode="barycentre")
    save_merged_grid(merged_fusion, path_data, zone, cle_geo, SOURCE, SEUIL_OBS,SEUIL_SPECIES)
    #plot_fusionned_grid(merged_fusion, var_obs='nombreObs', title=f"{zone} Fusion", figsize=(10,10), log_scale=True)
    df_biodiv_fusion = apply_fusion_to_biodiv(df_final, merged_fusion, cle_geo=cle_geo, cle_ID=CLE_ID, var_obs='nombreObs')
    save_fusionned_biodiv(df_biodiv_fusion, SOURCE, zone, cle_geo, path_data, SEUIL_OBS,SEUIL_SPECIES)
    print("🎉 Fusion OK")

# ================================================================
# 5️⃣ RUN PIPELINE FOR ALL COUNTRIES
# ================================================================
for country in COUNTRIES:
    process_country(country)

    # ---------- VIDER LA MÉMOIRE ----------
    print(f"🗑️ Nettoyage mémoire après {country}...")
    # Supprimer variables globales lourdes si présentes
    vars_to_delete = ['df_biodiv_clean', 'df_with_grid', 'df_final', 'merged_fusion', 'df_biodiv_fusion', 'grid']
    for var in vars_to_delete:
        if var in globals():
            del globals()[var]

print("\n🏁🏁🏁 TOUT EST TERMINÉ AVEC SUCCÈS 🏁🏁🏁")
